In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('email.csv')

In [ ]:
df.sample(5)

In [ ]:
df.shape

In [ ]:
#1. data cleaning
#2 exploratitary data analysis
#3 text preprocessing
# 4 model building
# 5 evaluation
# 6 improvement
# 7 website
# 8 deploy


In [ ]:
 ##data cleaning

In [ ]:
df.info()


In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()

In [ ]:
df['Category']=encoder.fit_transform(df['Category'])


In [ ]:
df.head()

In [ ]:
#check missing values 
df.isnull().sum()

In [ ]:
#check for duplicate values
df.duplicated().sum()

In [ ]:
#remove duplicates
df=df.drop_duplicates(keep='first')
df.duplicated().sum()

##2.EDA

In [ ]:
# Get indices of rows where category == 2
indices_to_drop = df[df['Category'] == 2].index

# Drop these rows (inplace=True modifies the original DataFrame)
df.drop(indices_to_drop, inplace=True)

In [ ]:
# Correct syntax to filter and count rows
df[df['Category'] == 0].count()


In [ ]:
import matplotlib.pyplot as plt
plt.pie(df['Category'].value_counts(),labels=['ham','spam'],autopct="%0.3f")
plt.show()


#data is imbalanced

In [ ]:
#calculating  chars words sentences etc

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
#num of chars
df['num_characters']=df['Message'].apply(len)
df.head()

In [ ]:
#num words
df['num_words']=df['Message'].apply(lambda x:len(nltk.word_tokenize(x)))
df.head()


In [ ]:
#num sentences
df['num_sentences']=df['Message'].apply(lambda x:len(nltk.sent_tokenize(x)))
df.head()


In [ ]:
df[['num_characters','num_words','num_sentences']].describe()

In [ ]:
df[df['Category']==1][['num_characters','num_words','num_sentences']].describe()

In [ ]:
df[df['Category']==0][['num_characters','num_words','num_sentences']].describe()

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords.words('english')

## data preprocessing  
1 lower case
2 tokenization
3 removing special characters 
4 removing stop words and punction
5 stemming

In [ ]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Make sure to download the necessary NLTK data files before running:
# nltk.download('punkt')
# nltk.download('stopwords')

ps = PorterStemmer()

def transform_text(text):
    # Lowercase the text
    text = text.lower()
    
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    
    # Remove non-alphanumeric tokens
    tokens = [token for token in tokens if token.isalnum()]
    
    # Remove stopwords and punctuation
    tokens = [token for token in tokens if token not in stopwords.words('english') and token not in string.punctuation]
    
    # Apply stemming
    tokens = [ps.stem(token) for token in tokens]
    
    return " ".join(tokens)


In [ ]:
transform_text('are u intrested in winning a lotterary?')

In [ ]:
df['transformed_text']=df['Message'].apply(transform_text)

In [ ]:
df.head()

## model

## Model Building — Multinomial Naive Bayes

We use MultinomialNB because CountVectorizer produces word-count features.
The vectorizer is fitted only on the training text to avoid vocabulary leakage.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, classification_report


In [ ]:
# Keep the text as text while splitting.
# This lets us fit CountVectorizer only on the training data.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['transformed_text'],
    df['Category'],
    test_size=0.2,
    random_state=2,
    stratify=df['Category']
)


In [ ]:
# Bag of Words
cv = CountVectorizer()

# Fit vocabulary on training data only
X_train = cv.fit_transform(X_train_text)

# Use the same vocabulary to transform test data
X_test = cv.transform(X_test_text)

print('Training matrix shape:', X_train.shape)
print('Testing matrix shape:', X_test.shape)


### Train Multinomial Naive Bayes


In [ ]:
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

y_pred = mnb.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))


## Test New Messages

The new message must go through the exact same preprocessing and vectorization steps used during training.


In [ ]:
# Test message 1
new_email = 'Your free prize is waiting... click here!'

processed_email = transform_text(new_email)
X_new = cv.transform([processed_email])
prediction = mnb.predict(X_new)[0]

print('Message:', new_email)
print('Prediction:', 'spam' if prediction == 1 else 'ham')


In [ ]:
# Test message 2
new2email = 'how are you'

new2transformed = transform_text(new2email)
X_new2 = cv.transform([new2transformed])
prediction2 = mnb.predict(X_new2)[0]

print('Message:', new2email)
print('Prediction:', 'spam' if prediction2 == 1 else 'ham')


## Save Model for Deployment

These two files are what the Streamlit app will load:
- `model.pkl` — trained MultinomialNB model
- `vectorizer.pkl` — fitted CountVectorizer


In [ ]:
import pickle

with open('model.pkl', 'wb') as file:
    pickle.dump(mnb, file)

with open('vectorizer.pkl', 'wb') as file:
    pickle.dump(cv, file)

print('model.pkl and vectorizer.pkl saved successfully.')


## Deployment Notes

The Streamlit application must use the same `transform_text()` function as training.
It should load `model.pkl` and `vectorizer.pkl`, preprocess the user's message, transform it with the saved vectorizer, and then call `mnb.predict()`.
